# Repeatability: aggregate report

Accumulated pose repeatability over many runs. The headline is the
mean within-run RP (ISO 9283) across runs with a t-based 95 %
confidence interval; the pooled cross-session RP is reported
alongside when all runs share one pose. `GROUP = 'all'` summarises
repeatability at every pose tested; `MATCH_CYCLES` filters to one
iteration count.

Protocol: 30 cycles per run (ISO 9283 RP is defined over a 30-point
cluster), 3 runs or more, re-homed between runs.

*Note: the kernel imports `volcaniarm_calibration` through a `.pth` file; restart the kernel after changing the package code. Every figure is also saved to `notebooks/figures/` as a 300 dpi PNG and a vector PDF, ready for the thesis.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volcaniarm_calibration.analysis import (
    load_runs, select_comparable_runs, filter_runs_by_goals,
    filter_runs_by_cycles, concat_runs, mount_key,
    apply_style, save_fig, run_short, per_axis_residuals_mm,
    PRIMARY, ACCENT, RUN_COLORS, FIG_FULL, FIG_TALL, FIG_SQUARE,
    summary, repeatability_iso9283, per_point_repeatability,
    threshold_zone,
)
apply_style()

TEST_NAME = 'repeatability'

GROUP = 'pose'           # 'pose' = same-pose averaging; 'all' = every pose pooled
POSE = None              # (y, z) used by GROUP='pose'; None = pose with most runs
MATCH_CYCLES = None      # e.g. 30 to keep only 30-cycle runs; None = include all
RUN_DIRS = None          # pin the exact run set for final thesis figures
ALLOW_MOUNT_KEYS = None  # merge legacy mount keys known to be identical
MIN_RUNS = 3             # protocol target; fewer runs still compute

all_runs = load_runs(TEST_NAME, run_dirs=RUN_DIRS)
HAVE = bool(all_runs)
if not HAVE:
    print('No completed runs on disk; record some from the calibration '
          'dashboard first.')

if HAVE:
    by_target = {}
    for r in all_runs:
        key = tuple(round(float(v), 3) for v in r['config']['goals'][0])
        by_target.setdefault(key, []).append(r)
    print('Targets on disk:')
    for key, rs in sorted(by_target.items()):
        print(f'  y={key[0]:+.3f} z={key[1]:.3f}: {len(rs)} run(s), '
              f'latest {rs[-1]["config"].get("run_id")}')
    print()

    if MATCH_CYCLES is not None:
        n_before = len(all_runs)
        all_runs = filter_runs_by_cycles(all_runs, MATCH_CYCLES)
        print(f'Cycle filter: kept {len(all_runs)}/{n_before} runs '
              f'with num_cycles == {MATCH_CYCLES}')
        if not all_runs:
            raise RuntimeError('no runs left after the cycle filter')

    if GROUP == 'pose':
        runs = (filter_runs_by_goals(all_runs, [POSE]) if POSE is not None
                else all_runs)
        if not runs:
            raise RuntimeError(f'no completed runs at pose {POSE}; '
                               'see the list above')
        runs = select_comparable_runs(runs, allow_mount_keys=ALLOW_MOUNT_KEYS)
    elif GROUP == 'all':
        runs = select_comparable_runs(all_runs,
                                      allow_mount_keys=ALLOW_MOUNT_KEYS,
                                      match_goals=False)
    else:
        raise ValueError(f"unknown GROUP {GROUP!r}")
    df = concat_runs(runs)
    n_targets = df[['goal_y', 'goal_z']].drop_duplicates().shape[0]
    goal_y, goal_z = runs[-1]['config']['goals'][0]
    run_ids = list(dict.fromkeys(df['run_id']))

    DIMS = ('det_ee_y', 'det_ee_z')
    rows, per_run_rp = [], []
    for run_id, g in df.groupby('run_id', sort=False):
        rep = repeatability_iso9283(g, dims=DIMS)
        rp_mm = rep['RP_m'] * 1000.0
        per_run_rp.append(rp_mm)
        cfg = next(r['config'] for r in runs
                   if r['config'].get('run_id') == run_id)
        py, pz = cfg['goals'][0]
        rows.append({'run': run_short(run_id),
                     'target': f'({py:+.2f}, {pz:.3f})',
                     'cycles': cfg.get('num_cycles'),
                     'n': rep['n'],
                     'RP [mm]': round(rp_mm, 2),
                     'worst [mm]': round(rep['worst_m'] * 1000.0, 2),
                     'std Y [mm]': round(rep['per_dim_std'][0] * 1000.0, 2),
                     'std Z [mm]': round(rep['per_dim_std'][1] * 1000.0, 2)})
    per_run_table = pd.DataFrame(rows)

    if n_targets == 1:
        print(f'Target: y = {goal_y:.3f} m, z = {goal_z:.3f} m')
    else:
        print(f'Scope: all poses pooled ({n_targets} poses)')
    print(f'Runs aggregated: {len(runs)}'
          + ('' if len(runs) >= MIN_RUNS else
             f'  (below the protocol target of {MIN_RUNS})'))
    display(per_run_table)

## Headline statistics

Within-run RP is the ISO-comparable number; the pooled figure adds
session effects (re-homing, camera relock) and is pessimistic. The
weeding application targets roughly 10 mm.

In [ ]:
if HAVE:
    rp_stats = summary(np.asarray(per_run_rp) / 1000.0).in_mm()

    if rp_stats.n >= 2:
        print(f'Within-run RP (ISO 9283):  {rp_stats.mean:6.2f} mm '
              f'+/- {rp_stats.ci95:.2f} mm (95 % CI, {rp_stats.n} runs)')
    else:
        print(f'Within-run RP (ISO 9283):  {rp_stats.mean:6.2f} mm '
              f'(single run; no across-run CI)')
    if n_targets == 1:
        pooled = repeatability_iso9283(df, dims=DIMS)
        print(f'Pooled cross-session RP:   '
              f'{pooled["RP_m"] * 1000:6.2f} mm '
              f'({pooled["n"]} cycles, includes re-homing and camera '
              f'relock)')
        print(f'Worst distance to pooled centroid: '
              f'{pooled["worst_m"] * 1000:6.2f} mm')
    else:
        pooled = None
        print('Pooled cross-session RP skipped: the aggregated runs '
              'target different poses; see the per-pose breakdown below.')
    print()
    print(f'Weeding zone of within-run RP: '
          f'{threshold_zone(rp_stats.mean)}')

## Per-pose breakdown

One row per commanded pose: pooled RP, mean within-run RP and sample
counts.

In [ ]:
if HAVE:
    breakdown = per_point_repeatability(df)
    display(breakdown)

## Attained-position clusters

Left: cycles about their own run centroid (the ISO 9283 view; circle
at the mean within-run RP). Right: the same cycles about the pooled
centroid, exposing between-session shifts. Colour identifies the
run.

In [ ]:
if HAVE:
    fig, axes = plt.subplots(1, 2, figsize=(6.3, 3.4),
                             sharex=True, sharey=True)
    pooled_cy = df['det_ee_y'].mean() * 1000.0
    pooled_cz = df['det_ee_z'].mean() * 1000.0
    for i, rid in enumerate(run_ids):
        g = df[df['run_id'] == rid]
        y_mm = g['det_ee_y'].to_numpy() * 1000.0
        z_mm = g['det_ee_z'].to_numpy() * 1000.0
        c = RUN_COLORS[i % len(RUN_COLORS)]
        axes[0].plot(y_mm - y_mm.mean(), z_mm - z_mm.mean(), 'o', ms=6,
                     color=c, mec='white', mew=0.8, alpha=0.9,
                     label=run_short(rid))
        axes[1].plot(y_mm - pooled_cy, z_mm - pooled_cz, 'o', ms=6,
                     color=c, mec='white', mew=0.8, alpha=0.9)
    theta = np.linspace(0, 2 * np.pi, 200)
    if not np.isnan(rp_stats.mean):
        axes[0].plot(rp_stats.mean * np.cos(theta),
                     rp_stats.mean * np.sin(theta), '-', lw=1.4,
                     color='#555555',
                     label=f'mean within-run RP ({rp_stats.mean:.2f} mm)')
    rp_pool_mm = (pooled['RP_m'] * 1000.0 if pooled is not None
                  else float('nan'))
    if not np.isnan(rp_pool_mm):
        axes[1].plot(rp_pool_mm * np.cos(theta),
                     rp_pool_mm * np.sin(theta), '-', lw=1.4,
                     color='#555555',
                     label=f'pooled RP ({rp_pool_mm:.2f} mm)')
    for ax, title in zip(axes, ('About each run centroid',
                                'About the pooled centroid')):
        ax.plot(0, 0, '+', ms=10, color='#333333', mew=2)
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel('Y offset  [mm]')
        ax.set_title(title, fontsize=11)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(loc='best', fontsize=8)
    axes[0].set_ylabel('Z offset  [mm]')
    save_fig(fig, 'repeatability_aggregate/clusters')

## RP per run

One bar per run with the worst cycle marked; an outlier session is
visible at a glance.

In [ ]:
if HAVE:
    worst_by_run = [
        float(np.linalg.norm(
            g[['det_ee_y', 'det_ee_z']].to_numpy()
            - g[['det_ee_y', 'det_ee_z']].to_numpy().mean(axis=0),
            axis=1).max()) * 1000.0
        for _, g in df.groupby('run_id', sort=False)]
    xs = np.arange(len(per_run_rp))
    fig, ax = plt.subplots(figsize=FIG_FULL)
    ax.bar(xs, per_run_rp,
           color=[RUN_COLORS[i % len(RUN_COLORS)] for i in xs],
           edgecolor='white', width=0.55, label='RP')
    ax.plot(xs, worst_by_run, 'v', ms=8, color='#333333',
            label='worst cycle')
    if not np.isnan(rp_stats.mean):
        ax.axhline(rp_stats.mean, color='#555555', ls='--', lw=1.2,
                   label='mean RP')
    ax.set_xticks(xs)
    ax.set_xticklabels([run_short(r) for r in run_ids], rotation=15)
    ax.set_xlabel('run (date time)')
    ax.set_ylabel('[mm]')
    ax.set_title('Repeatability per run')
    ax.legend(loc='best')
    save_fig(fig, 'repeatability_aggregate/rp_per_run')

## Distance-to-centroid distribution

Histogram of each cycle's distance to its own run centroid with the
mean within-run RP marked.

In [ ]:
if HAVE:
    d_all = np.concatenate([
        np.linalg.norm(
            g[['det_ee_y', 'det_ee_z']].to_numpy()
            - g[['det_ee_y', 'det_ee_z']].to_numpy().mean(axis=0),
            axis=1) * 1000.0
        for _, g in df.groupby('run_id', sort=False)])
    fig, ax = plt.subplots(figsize=FIG_FULL)
    ax.hist(d_all, bins=15, color=PRIMARY, edgecolor='white', alpha=0.85)
    if not np.isnan(rp_stats.mean):
        ax.axvline(rp_stats.mean, color='#d04b4b', ls='--', lw=1.5,
                   label=f'mean within-run RP ({rp_stats.mean:.2f} mm)')
    ax.set_xlabel('distance to run centroid  [mm]')
    ax.set_ylabel('cycles')
    ax.set_title('Distribution of distances to the run centroid')
    ax.legend(loc='best')
    save_fig(fig, 'repeatability_aggregate/distance_hist')

## Within-run drift across the campaign

Distance to the run centroid vs cycle with a linear trend per run;
a recurring slope across sessions indicates a systematic drift
mechanism rather than a one-off.

In [ ]:
if HAVE:
    fig, ax = plt.subplots(figsize=FIG_TALL)
    for i, (rid, g) in enumerate(df.groupby('run_id', sort=False)):
        pos = g[['det_ee_y', 'det_ee_z']].to_numpy()
        d = np.linalg.norm(pos - pos.mean(axis=0), axis=1) * 1000.0
        xc = np.arange(1, len(d) + 1)
        c = RUN_COLORS[i % len(RUN_COLORS)]
        ax.plot(xc, d, 'o', ms=5, color=c, mec='white', mew=0.5,
                alpha=0.9, label=run_short(rid))
        if len(d) > 2:
            slope, intercept = np.polyfit(xc, d, 1)
            ax.plot(xc, slope * xc + intercept, '--', lw=1.2, color=c)
    ax.set_xlabel('cycle')
    ax.set_ylabel('distance to run centroid  [mm]')
    ax.set_title('Within-run drift per session')
    ax.legend(loc='best', fontsize=8)
    save_fig(fig, 'repeatability_aggregate/drift')

## Repeatability across sessions

Each run's RP in recording order; stability of the repeatability
itself over the campaign.

In [ ]:
if HAVE:
    xs = np.arange(len(per_run_rp))
    fig, ax = plt.subplots(figsize=FIG_FULL)
    ax.plot(xs, per_run_rp, 'o-', ms=7, lw=1.4, color=PRIMARY,
            mec='white', mew=0.8)
    if not np.isnan(rp_stats.mean):
        ax.axhline(rp_stats.mean, color='#555555', ls='--', lw=1.2,
                   label='mean RP')
        ax.legend(loc='best')
    ax.set_xticks(xs)
    ax.set_xticklabels([run_short(r) for r in run_ids], rotation=15)
    ax.set_xlabel('run (date time)')
    ax.set_ylabel('RP  [mm]')
    ax.set_title('RP across sessions')
    save_fig(fig, 'repeatability_aggregate/sessions')

## Interpretation

Quote the mean within-run RP with its t-based confidence interval as
the ISO 9283 repeatability; the pooled figure bounds what the
application sees across working sessions and is labelled as such. A
markedly elliptical cluster (per-axis std ratio far from 1) points at
a dominant mechanical axis. Compare against the roughly 10 mm weeding
tolerance for the application argument.